Baselines: Random and Markov Chain

In [1]:
import os
import random
import numpy as np
import pandas as pd
import pretty_midi
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)
from src.generation.midi_export import write_notes_to_midi, validate_midi

raw_dir = os.path.join('data', 'raw_midi', 'maestro-v3.0.0')
fallback_dir = os.path.join('data', 'maestro-v3.0.0')
metadata_file = 'maestro-v3.0.0.csv'
data_dir = raw_dir if os.path.exists(os.path.join(raw_dir, metadata_file)) else fallback_dir
metadata = pd.read_csv(os.path.join(data_dir, metadata_file))
train_meta = metadata[metadata['split'] == 'train']

def extract_pitch_sequence(midi_path):
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
    except Exception:
        return [], []
    notes = sorted([n for inst in pm.instruments for n in inst.notes], key=lambda x: x.start)
    pitches = [n.pitch for n in notes]
    durations = [n.end - n.start for n in notes]
    return pitches, durations

num_pitches = 128
counts = np.ones((num_pitches, num_pitches), dtype=np.float32)
durations = []
for _, row in train_meta.iterrows():
    midi_file = os.path.join(data_dir, row['midi_filename'])
    pitches, durs = extract_pitch_sequence(midi_file)
    if len(pitches) < 2:
        continue
    durations.extend(durs)
    for a, b in zip(pitches[:-1], pitches[1:]):
        counts[a, b] += 1
transition = counts / counts.sum(axis=1, keepdims=True)
if not durations:
    durations = [0.25, 0.5, 1.0]

def generate_random(notes=200):
    durations_set = [0.25, 0.5, 1.0, 2.0]
    t = 0.0
    out = []
    for _ in range(notes):
        pitch = random.randint(21, 108)
        dur = random.choice(durations_set)
        out.append((pitch, t, t + dur, 80))
        t += dur
    return out

def generate_markov(notes=200):
    pitch = random.randint(21, 108)
    t = 0.0
    out = []
    for _ in range(notes):
        dur = float(random.choice(durations))
        out.append((pitch, t, t + dur, 80))
        t += dur
        probs = transition[pitch]
        pitch = int(np.random.choice(np.arange(num_pitches), p=probs))
        if pitch < 21 or pitch > 108:
            pitch = random.randint(21, 108)
    return out

out_dir = os.path.join('outputs', 'generated_midis', 'baselines')
os.makedirs(out_dir, exist_ok=True)
random_path = os.path.join(out_dir, 'random_1.mid')
markov_path = os.path.join(out_dir, 'markov_1.mid')
write_notes_to_midi(generate_random(), random_path)
write_notes_to_midi(generate_markov(), markov_path)

print('Random valid:', validate_midi(random_path))
print('Markov valid:', validate_midi(markov_path))

Random valid: True
Markov valid: True
